# 01 · LoCoMo 데이터셋 — 벤치마크를 의심하며 읽기

이 챕터가 답하는 질문:

1. LoCoMo는 무엇이고 왜 agent memory 검증에 쓰이는가?
2. 데이터는 실제로 어떻게 생겼는가?
3. QA 카테고리 1~5는 각각 무엇을 시험하는가? — **숫자만 주어지므로 이름은 데이터로 검증한다**
4. 이 데이터의 함정(때)은 무엇이고, 그것이 실험에 어떤 영향을 주는가?

> **전제**: `uv run scripts/fetch_data.py`로 데이터를 받아둔 상태여야 한다.
> 이 노트북의 모든 셀은 API 키 없이, 수 초 안에 실행된다.
>
> 이 장의 모든 주장은 `tests/test_locomo.py`에 테스트로 고정되어 있다 —
> 노트북은 이야기를 하고, 테스트는 사실을 지킨다.

## 1. LoCoMo란

**LoCoMo** (Long Conversational Memory)는 Maharana et al., *Evaluating Very
Long-Term Conversational Memory of LLM Agents* (ACL 2024, Snap Research)에서
공개한 벤치마크다. 질문은 단순하다: **LLM 에이전트는 수개월에 걸친 대화를
기억할 수 있는가?**

- 두 사람이 여러 달 동안 나눈 **초장기 대화** 10편. 각 대화는 수십 개의
  *세션*(날짜가 찍힌 만남)으로 구성된다.
- 페르소나 + 시간적 사건 그래프를 가진 LLM 생성 파이프라인으로 만들고
  사람이 검수했다. 이미지를 공유하는 턴도 있다(캡션으로 제공).
- 대화가 끝나면 **QA 시험지**로 기억을 시험한다: "Melanie가 언제 캠핑을
  갔지?" — 정답과 근거 발화(evidence)가 어노테이션돼 있다.

우리가 재현하려는 MemoryOS(arXiv:2506.06326)를 포함해, agent memory
연구들이 이 벤치마크로 성능을 보고한다.

### 데이터 계보 — 왜 원 출처에서 받는가

우리는 데이터를 MemoryOS repo의 사본이 아니라 **원 출처**
(snap-research/locomo)에서 받는다. `scripts/fetch_data.py`가 커밋 SHA로
*시점*을, SHA-256 체크섬으로 *내용*을 이중으로 고정한다. 논문 repo의
사본은 "한 논문의 스냅샷"이지 벤치마크가 아니다.
(검증 결과 MemoryOS의 사본은 원본과 바이트 단위로 동일했다 — 그래도
계보는 원 출처로 두는 것이 맞다.)

In [1]:
from collections import Counter

from memlab.data import Category, load_locomo

samples = load_locomo()

print(f"샘플(대화):  {len(samples)}")
print(f"턴(발화):    {sum(s.num_turns for s in samples):,}")
print(f"QA(시험문제): {sum(len(s.qa) for s in samples):,}")

샘플(대화):  10
턴(발화):    5,882
QA(시험문제): 1,986


## 2. 대화의 생김새

샘플 하나 = 두 화자의 대화 전체. 대화는 세션들로, 세션은 턴들로 구성된다.

```
Sample (conv-26: Caroline & Melanie)
├── Session 1  ("1:56 pm on 8 May, 2023")
│   ├── Turn D1:1  Caroline: "Hey Mel! ..."
│   ├── Turn D1:2  Melanie:  "Hey Caroline! ..."
│   └── ...
├── Session 2  (다른 날)
└── ...
```

- `dia_id`(예: `D3:7` = 세션 3의 7번째 발화)는 QA의 evidence가 참조하는 주소다.
- 턴의 ~21%는 이미지 공유 턴 — 모델에게는 `blip_caption`(이미지 캡션)으로 보인다.

In [2]:
s = samples[0]
print(f"{s.sample_id}: {s.speaker_a} & {s.speaker_b}")
print(f"세션 {len(s.sessions)}개, 턴 {s.num_turns}개")
print(f"기간: {s.sessions[0].date_time}  →  {s.sessions[-1].date_time}")
print()
for t in s.sessions[0].turns[:4]:
    print(f"  [{t.dia_id}] {t.speaker}: {t.text[:70]}")

# 이미지 공유 턴의 예
img_turn = next(t for sess in s.sessions for t in sess.turns if t.has_image)
print(f"\n이미지 턴 [{img_turn.dia_id}] {img_turn.speaker}: {img_turn.text[:50]}")
print(f"  캡션: {img_turn.blip_caption}")

n_img = sum(1 for sm in samples for sess in sm.sessions for t in sess.turns if t.has_image)
print(f"\n이미지 턴 합계: {n_img} / 5,882 (~{n_img / 5882:.0%})")

conv-26: Caroline & Melanie
세션 19개, 턴 419개
기간: 1:56 pm on 8 May, 2023  →  9:55 am on 22 October, 2023

  [D1:1] Caroline: Hey Mel! Good to see you! How have you been?
  [D1:2] Melanie: Hey Caroline! Good to see you! I'm swamped with the kids & work. What'
  [D1:3] Caroline: I went to a LGBTQ support group yesterday and it was so powerful.
  [D1:4] Melanie: Wow, that's cool, Caroline! What happened that was so awesome? Did you

이미지 턴 [D1:5] Caroline: The transgender stories were so inspiring! I was s
  캡션: a photo of a dog walking past a wall with a painting of a woman

이미지 턴 합계: 1226 / 5,882 (~21%)


## 3. QA — 시험지, 그리고 다섯 카테고리

각 QA는 `question / answer / evidence / category`를 가진다. category는
**1~5의 숫자로만** 주어진다 — 이름은 논문을 근거로 *추정*할 수 있지만,
추정을 그대로 믿지 말고 **데이터로 검증**하자. 이것이 이 장의 핵심 연습이다.

LoCoMo 논문이 말하는 다섯 유형: single-hop(근거 1개), multi-hop(근거 여러 개
종합), temporal(시간 추론), open-domain(대화 + 상식 추론), adversarial(답할 수
없는 질문 — 안 속고 버티는 게 정답).

먼저 카테고리별 예시를 눈으로 보고:

In [3]:
seen = set()
for sm in samples:
    for q in sm.qa:
        if q.category not in seen:
            seen.add(q.category)
            print(f"[cat{int(q.category)}] Q: {q.question}")
            print(f"        A: {q.answer!r}   evidence={list(q.evidence)}")
            if q.adversarial_answer:
                print(f"        (adversarial_answer: {q.adversarial_answer!r})")
            print()
    if len(seen) == 5:
        break

[cat2] Q: When did Caroline go to the LGBTQ support group?
        A: '7 May 2023'   evidence=['D1:3']

[cat3] Q: What fields would Caroline be likely to pursue in her educaton?
        A: 'Psychology, counseling certification'   evidence=['D1:9', 'D1:11']

[cat1] Q: What did Caroline research?
        A: 'Adoption agencies'   evidence=['D2:8']

[cat4] Q: What did the charity race raise awareness for?
        A: 'mental health'   evidence=['D2:2']

[cat5] Q: What did Caroline realize after her charity race?
        A: None   evidence=['D2:3']
        (adversarial_answer: 'self-care is important')



In [4]:
# 카테고리 이름을 데이터로 검증한다:
#  - multi-hop이라면 evidence가 여러 개여야 한다
#  - temporal이라면 답이 날짜여야 한다
import re

date_pat = re.compile(
    r"\b(19|20)\d{2}\b|january|february|march|april|may|june"
    r"|july|august|september|october|november|december",
    re.I,
)

print("cat | QA수 | 평균 evidence | evidence>=2 | 답에 날짜")
print("----+------+--------------+-------------+----------")
for cat in Category:
    qs = [q for sm in samples for q in sm.qa if q.category == cat]
    evs = [len(q.evidence) for q in qs]
    multi = sum(1 for e in evs if e >= 2) / len(evs)
    dates = sum(1 for q in qs if q.answer and date_pat.search(q.answer)) / len(qs)
    print(
        f"  {int(cat)} | {len(qs):4d} |     {sum(evs) / len(evs):.2f}     |"
        f"    {multi:4.0%}     |   {dates:4.0%}   <- {cat.name}"
    )

cat | QA수 | 평균 evidence | evidence>=2 | 답에 날짜
----+------+--------------+-------------+----------
  1 |  282 |     3.13     |     98%     |     2%   <- MULTI_HOP
  2 |  321 |     1.17     |     12%     |    80%   <- TEMPORAL
  3 |   96 |     2.08     |     49%     |     0%   <- OPEN_DOMAIN
  4 |  841 |     1.07     |      5%     |     1%   <- SINGLE_HOP
  5 |  446 |     1.03     |      3%     |     0%   <- ADVERSARIAL


표를 읽으면 매핑이 데이터에서 그대로 드러난다:

| cat | 증거 | 결론 |
|---|---|---|
| 1 | 평균 evidence **3.13개**, 98%가 2개 이상 | **MULTI_HOP** — 여러 발화를 종합해야 답이 나옴 |
| 2 | 답변의 **80%가 날짜** | **TEMPORAL** — "언제?"를 묻는다 |
| 3 | 96개뿐, 절반이 다중 근거, 질문이 추론형("~할 것 같은가?") | **OPEN_DOMAIN** — 대화 + 상식 추론 |
| 4 | evidence ~1개, **최대 분포(841개)** | **SINGLE_HOP** — 한 발화에서 바로 답 |
| 5 | 446개 중 **444개가 정답 없음**, 대신 `adversarial_answer` 보유 | **ADVERSARIAL** — 일어난 적 없는 일을 묻는다 |

### adversarial이 특별한 이유

cat5는 "Caroline이 charity race 후에 뭘 깨달았지?" 같은 **일어나지 않은 일**을
묻는다. `adversarial_answer`는 *그럴듯한 오답*이다 — 시스템이 지어내면 이 오답에
가까운 답을 하게 된다. 즉 cat5는 기억력이 아니라 **환각 저항**을 시험한다.

채점에도 영향을 준다: 정답 문자열이 없으므로 F1/BLEU를 그대로 계산할 수 없다.
그래서 우리 채점기는 `--exclude-adversarial` 옵션을 갖는다 (챕터 02에서).

## 4. 데이터의 때 — 실제 벤치마크는 깨끗하지 않다

전수 조사에서 찾은 함정들:

1. **유령 세션 16개** — `session_N_date_time`은 있는데 `session_N` 본문이 없다.
   세션 번호가 연속이라고 가정하는 코드는 여기서 깨진다.
2. **깨진 evidence 9개** — 한 문자열에 id 여러 개(`'D9:1 D4:4 D4:6'`),
   잘린 id(`'D'`), 0 패딩(`'D30:05'`) 등.
3. **int인 answer 6개** — 연도 등. (로더가 str로 정규화하는 유일한 값.)

**로더의 원칙: 고치지 말고, 보존하고, 문서화하라.** 조용히 '고쳐주는' 로더는
나중에 원본 기준의 분석과 어긋나는 원인이 된다. 대신 `tests/test_locomo.py`의
*회계 테스트*가 "버리는 것의 목록이 완전함"을 강제한다 — 데이터에 우리가 모르는
키가 나타나면 테스트가 실패하며 결정을 요구한다.

In [5]:
# 깨진 evidence 9개를 직접 보자
for sm in samples:
    ids = {t.dia_id for sess in sm.sessions for t in sess.turns}
    for q in sm.qa:
        for e in q.evidence:
            if e not in ids:
                print(f"{sm.sample_id}  cat{int(q.category)}  evidence={e!r}")

conv-26  cat1  evidence='D8:6; D9:17'
conv-42  cat1  evidence='D10:19'
conv-42  cat4  evidence='D'
conv-43  cat1  evidence='D:11:26'
conv-47  cat1  evidence='D4:36'
conv-49  cat3  evidence='D9:1 D4:4 D4:6'
conv-49  cat3  evidence='D22:1 D22:2 D9:10 D9:11'
conv-49  cat3  evidence='D21:18 D21:22 D11:15 D11:19'
conv-50  cat2  evidence='D30:05'


## 5. 화자 구조, 그리고 원본 전처리의 결함

모든 샘플은 정확히 **두 화자**이고, 세션 안에서 **완벽하게 번갈아** 말한다
(연속 동일 화자 0/5,610). 여기까지는 깨끗하다. 함정은 다른 곳에 있다:
**세션의 45%(124/272)는 speaker_b가 먼저 말한다.**

왜 문제인가? MemoryOS는 사용자↔비서 대화용 시스템이라, 원본 실험 코드
(`eval/main_loco_parse.py`의 `process_conversation`)는 친구 대화를 이렇게 접는다:

> a가 말하면 → 새 쌍의 사용자 칸에. b가 말하면 → **가장 최근 쌍**의 응답 칸에.

세션이 b로 시작하면 그 발화는 **이전 세션의 마지막 쌍**에 들어간다:

- 그 칸이 차 있으면 → **이전 발화를 덮어써서 유실** (61턴)
- 비어 있으면 → 이전 세션의 발화와 짝지어지고 **이전 세션의 타임스탬프**를 받는다 (57턴)

"세션은 항상 a가 먼저 말한다"는 검증 안 된 가정 하나가 만든 결함이다.
아래 셀이 원본 알고리즘을 시뮬레이션해 피해 규모를 잰다:

In [6]:
# 원본 process_conversation의 쌍 접기를 시뮬레이션
lost, mispaired = set(), set()
for sm in samples:
    processed = []  # {sess, filled, b_id}: 쌍의 상태만 추적
    for sess in sm.sessions:
        for t in sess.turns:
            if t.speaker == sm.speaker_a:
                processed.append({"sess": sess.index, "filled": False, "b_id": None})
            elif processed:
                if processed[-1]["filled"]:  # 이미 찬 응답 칸을 덮어씀
                    lost.add((sm.sample_id, processed[-1]["b_id"]))
                elif processed[-1]["sess"] != sess.index:  # 세션 경계를 넘은 짝
                    mispaired.add((sm.sample_id, t.dia_id))
                processed[-1]["filled"] = True
                processed[-1]["b_id"] = t.dia_id
            else:
                processed.append({"sess": sess.index, "filled": True, "b_id": t.dia_id})

affected = [
    (sm.sample_id, int(q.category), q.question[:55])
    for sm in samples
    for q in sm.qa
    if any((sm.sample_id, e) in lost or (sm.sample_id, e) in mispaired for e in q.evidence)
]
print(f"유실된 발화: {len(lost)}턴  /  잘못 짝지어진 발화: {len(mispaired)}턴")
print(f"유실/오염 턴을 근거로 요구하는 QA: {len(affected)}건")
print("\n예시 (temporal 질문이 잘못된 타임스탬프의 발화를 근거로 요구):")
for row in [a for a in affected if a[1] == 2][:3]:
    print("  ", row)

유실된 발화: 61턴  /  잘못 짝지어진 발화: 57턴
유실/오염 턴을 근거로 요구하는 QA: 87건

예시 (temporal 질문이 잘못된 타임스탬프의 발화를 근거로 요구):
   ('conv-26', 2, 'When did Melanie go camping in July?')
   ('conv-30', 2, 'When did Gina launch an ad campaign for her store?')
   ('conv-30', 2, 'When did Gina start being recognized by fashion editors')


## 6. 정리

- LoCoMo = 두 사람의 수개월짜리 대화 10편 + QA 1,986문제. **기억의 다섯 능력**
  (single-hop / multi-hop / temporal / open-domain 추론 / 환각 저항)을 시험한다.
- 카테고리 이름은 추정하지 않고 **데이터로 검증**했다 (evidence 개수, 날짜 비율).
- 실제 벤치마크에는 때가 있다 (유령 세션, 깨진 evidence). 로더는 고치지 않고
  보존 + 문서화하며, 테스트가 그 완전성을 강제한다.
- **원본 MemoryOS eval은 전처리 결함으로 61턴을 유실하고 57턴을 오염시킨다**
  (QA 87건의 근거에 영향). baseline 재현 시 이 동작까지 복제하고
  (`pairing="original"`), 수정 효과는 baseline 확정 후 별도 실험으로 잰다.

**다음 챕터(02)**: 답이 "맞았다"를 어떻게 정의하는가 — set-F1(원본 방식) vs
표준 F1 vs BLEU-1을 손으로 계산하며 채점기를 만든다.